<a href="https://colab.research.google.com/github/rhonaeloisa/bigdata/blob/lab3/big_data_lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
import csv
from io import StringIO
from pyspark import SparkContext
import pyspark

# Initialize the Spark Session
spark = SparkSession.builder.appName("Flood Control").getOrCreate()

sc = spark.sparkContext



In [ ]:
# Read the file
rdd_file = sc.textFile("/content/dpwh_flood_control_projects.csv")

# Get header
header = rdd_file.first()

#remove the header row
no_header = rdd_file.filter(lambda line: line != header)

#parse each line using csv.reader
#so commas inside quoted fields are handled correctly
def parse_csv(line):
  return next(csv.reader(StringIO(line)))

rdd_parsed = no_header.map(parse_csv)


In [ ]:
#HASH PARTITIONING

#convert each row into a key-value pair
#type of work (index 8) becomes key
rdd_pair = rdd_parsed.map(lambda x: (x[8], x))

#Apply hash partitioning. Make 4 partitions
hash_partition = rdd_pair.partitionBy(4)

#display the number of rows in each partition
print("Partition by Type of Work")
partition_counts = hash_partition.glom().map(lambda x: len(x)).collect()
print(partition_counts)
print()

#convert each row into a count of 1
type_of_work = hash_partition.mapValues(lambda x: 1)

#get total number of projects per type of work
type_of_work_project = type_of_work.reduceByKey(lambda a, b: a + b)

#sort the data by the project count in descending order
type_of_work_project_sorted = type_of_work_project.sortBy(lambda x: x[1], ascending=False)

#collect and display the total project count per type of work
type_of_work_project_sorted.collect()


Partition by Type of Work
[6608, 848, 1432, 967]



[('Construction of Flood Mitigation Structure', 6352),
 ('Construction of Revetment', 747),
 ('Construction of Slope Protection Structure', 739),
 ('Construction of Drainage Structure', 679),
 ('Rehabilitation / Major Repair of Flood Control Structure', 648),
 ('Construction of Dike', 225),
 ('Rehabilitation / Major Repair of Drainage Structure', 215),
 ('Construction of Flood Mitigation Facility', 170),
 ('Rehabilitation / Major Repair of Slope Protection Structure', 41),
 ('Rehabilitation / Major Repair of Flood Mitigation Facility', 12),
 ('Construction of Groundsill', 12),
 ('Construction of Retarding Basin', 5),
 ('Construction of Spur Dike', 3),
 ('Construction of Cutoff Channel', 2),
 ('Construction of Waterway', 2),
 ('Embankment', 1),
 ('Upgrading of Drainage Structure', 1),
 ('Repair/Maintenance of Flood Control Structures', 1)]

In [ ]:
from datetime import datetime

#custom partition by year
def date_partitioner(key):
    try:
        d = datetime.strptime(key, "%Y-%m-%d")
        if d.year < 2020:
            return 0
        elif d.year < 2022:
            return 1
        elif d.year < 2024: return 2
        else: return 3
    except:
        return 0

rdd_key = rdd_parsed.map(lambda cols: (cols[13], cols[7]))
rdd_partitioned = rdd_key.partitionBy(4, date_partitioner)

print("partition by year")
partition_counts = rdd_partitioned.glom().map(lambda x: len(x)).collect()
print(partition_counts)
print()

#extract year and count
year_counts = (
    rdd_partitioned
    .map(lambda x: (datetime.strptime(x[0], "%Y-%m-%d").year, 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

for year, count in sorted(year_counts):
    print(f"Total projects done in {year}: {count}")


partition by year
[0, 0, 6359, 3496]

Total projects in 2022: 2531
Total projects in 2023: 3828
Total projects in 2024: 3120
Total projects in 2025: 376


In [ ]:
# alisin yung mga rows na hindi naka digit
rdd_clean = rdd_parsed.filter(lambda x: len(x) > 12 and x[12].replace('.','',1).isdigit())

# rdd na yung cost yung key tapos yung whole row yung value
rdd_pair = rdd_clean.map(lambda x: (float(x[12]), x))

# hatiin sa 3 partitions
range_rdd = rdd_pair.sortByKey(numPartitions=3)

# i-print kung ilang rows meron sa partitions
print("Partition by Contract Cost (Range)")
partition_counts = range_rdd.glom().map(lambda x: len(x)).collect()
print(partition_counts)
print()

# rdd na tiga-link yung office sa cost
location_cost_pair = rdd_clean.map(lambda x: (x[5].strip(), float(x[12])))

# i-add lahat na cost per office
location_totals = location_cost_pair.reduceByKey(lambda a, b: a + b)

# i-sort yung office na may highest cost
final_result = location_totals.sortBy(lambda x: x[1], ascending=False)

# kunin lang yung top 10
final_result.take(10)

Partition by Contract Cost (Range)
[3646, 2789, 3392]



[('Bulacan 1st District Engineering Office', 29183182391.39999),
 ('North Manila District Engineering Office', 12768889763.279991),
 ('Cebu 7th District Engineering Office', 12755004902.119999),
 ('Bulacan 2nd District Engineering Office', 12399706689.519995),
 ('Metro Manila 1st District Engineering Office', 12352162536.09),
 ('Tarlac District Engineering Office', 11865023760.449997),
 ('La Union 2nd District Engineering Office', 9085915597.28),
 ('Isabela 4th District Engineering Office', 8842387406.539999),
 ('Mindoro Occidental District Engineering Office', 8720533485.880001),
 ('Mindoro Oriental District Engineering Office', 7840528391.699998),
 ('Abra District Engineering Office', 7652897214.749999),
 ('Isabela 3rd District Engineering Office', 6880615326.729998),
 ('Tarlac 2nd District Engineering Office', 6719892372.880001),
 ('Cavite 3rd District Engineering Office', 6446035875.359999),
 ('Butuan City District Engineering Office', 6300109823.440001),
 ('Albay 2nd District Engi